**Note:** Update the file paths to your own data location before running.

In [ ]:
from google.colab import drive

import pandas as pd
import operator
import numpy as np
from sklearn.metrics import r2_score
import random
import copy
import math
from scipy.stats import spearmanr
import matplotlib.pyplot as plt

In [ ]:
drive.mount('/content/drive')

file_path = "/content/drive/My Drive/GWL/r2_complexity/train219_val99_2obj of GP/train219_dataset.csv"
df = pd.read_csv(file_path)


FEATURE_COLUMNS = ['Soil', 'Slope', 'Rain', 'LULC', 'Lithology','Lineament density', 'Geomorphology', 'Drainage density']
# #FEATURE_COLUMNS = ['geomorphology','Rain','drainage density']
TARGET_COLUMN = 'Net Groundwater Recharge'

X = df[FEATURE_COLUMNS].values
y = df[TARGET_COLUMN].values
validation_labels = df['validation point categorical data'].values



# Filter dataframe to use selected features
#df_filtered = df[FEATURE_COLUMNS + [TARGET_COLUMN]].copy()

# Optional: filter out extreme target values (e.g., outliers above 300)
#df_filtered = df_filtered[df_filtered[TARGET_COLUMN] < 300]

# Extract X and y
#X = df_filtered[FEATURE_COLUMNS].values
#y = df_filtered[TARGET_COLUMN].values

#Inspect shapes and first few rows
#print("Shape of X:", X.shape)
#print("Shape of y:", y.shape)
#print("Sample features (X):", X[:3])
#print("Sample target values (y):", y[:3])

# Mapping: x0 -> 'Soil', x1 -> 'Slope', etc.
symbolic_variable_map = {f'x{i}': FEATURE_COLUMNS[i] for i in range(len(FEATURE_COLUMNS))}

print("Symbolic Variable Mapping:")
for symbol, name in symbolic_variable_map.items():
     print(f"{symbol} → {name}")

In [ ]:
def clamp(v, lo, hi):
    return max(lo, min(hi, v))

def safe_exp(x):
    # Prevent overflow
    return math.exp(clamp(x, -20, 20))

def safe_pow(x, y):
    # Prevent extreme exponents and huge bases
    base = abs(x) + 1e-6
    expn = clamp(y, -3, 3)
    return math.pow(base, expn)

# Protected division to avoid divide-by-zero
def protected_div(x, y):
    try:
        return x / y if abs(y) > 1e-6 else 1.0
    except:
        return 1.0

OPERATORS = {
    '+': (operator.add, 2),
    '-': (operator.sub, 2),
    '*': (operator.mul, 2),
    '/': (protected_div, 2),
    'neg': (lambda x: -x, 1),
    'sin': (math.sin, 1),
    'cos': (math.cos, 1),
    #'exp': (math.exp, 1),
    'log': (lambda x: math.log(abs(x) + 1e-6), 1),  # protected log
    'square': (lambda x: x * x, 1),
    'cube': (lambda x: x * x * x, 1),
    'exp': (safe_exp, 1),
    'pow': (safe_pow, 2),
}

class Node:
    def __init__(self, name, children=None):
        self.name = name
        self.children = children if children else []

    def evaluate(self, X_row, feature_map=None):
        if self.name in OPERATORS:
            func, arity = OPERATORS[self.name]
            if len(self.children) != arity:
                raise ValueError(f"Invalid expression: Operator '{self.name}' expects {arity} children, got {len(self.children)}")
            args = [child.evaluate(X_row, feature_map) for child in self.children]
            return func(*args)
        elif feature_map and self.name in feature_map:
            return X_row[feature_map[self.name]]
        elif isinstance(self.name, str) and self.name.startswith('x'):
            idx = int(self.name[1:])
            return X_row[idx]
        else:
            return float(self.name)

    # def count_nodes(self):
    #     return 1 + sum(child.count_nodes() for child in self.children)

    def to_string(self, feature_names=None):
        if self.name in OPERATORS:
            return f"{self.name}({', '.join(child.to_string(feature_names) for child in self.children)})"
        elif isinstance(self.name, str) and self.name.startswith('x') and feature_names:
            idx = int(self.name[1:])
            return feature_names[idx]
        else:
            return str(self.name)

    def __str__(self):
        return self.to_string()

    def weighted_complexity(self, parent_is_nonlinear=False, depth=1):
        LINEAR_OPS = {'+', '-', 'neg'}
        MILD_NONLINEAR_OPS = {'*'}
        STRONG_NONLINEAR_OPS = {'/', 'log', 'exp', 'pow', 'sin', 'cos', 'square', 'cube'}

        #feature=0
        if isinstance(self.name, str) and self.name.startswith('x'):
            cost = 0
            is_nonlinear = False
        elif self.name in LINEAR_OPS:
            cost = 1
            is_nonlinear = False
        elif self.name in MILD_NONLINEAR_OPS:
            cost = 1.25
            is_nonlinear = False
        elif self.name in STRONG_NONLINEAR_OPS:
            if self.name == '/':
                cost = 1.5
            elif self.name in ['square', 'cube']:
                cost = 1.5 if self.name == 'square' else 2
            elif self.name in ['sin', 'cos', 'log', 'exp']:
                cost = 1.25
            else:  # pow
                cost = 2
            is_nonlinear = True
        elif is_constant(self.name):
            cost = 0.5
            is_nonlinear = False
        else:
            cost = 1
            is_nonlinear = False

        # 2. Penalize ONLY problematic cases:
        # Very deep trees (>15 levels)
        depth_penalty = 0.1 * max(0, depth - 15)  # Applies to ALL nodes

        # Redundant nesting (sin(cos)), but not single nonlinear terms
        recursive_penalty = 0.05 if (parent_is_nonlinear and self.name in STRONG_NONLINEAR_OPS) else 0

        total = cost + depth_penalty + recursive_penalty
        for child in self.children:
            total += child.weighted_complexity(self.name in STRONG_NONLINEAR_OPS, depth + 1)
        return total

def is_constant(name):
    #Check if node is a numeric constant (handles negatives and floats)
    if isinstance(name, (int, float)):
        return True
    try:
        float(name)
        return True
    except (ValueError, TypeError):
        return False

def to_readable_string(self, var_map):
    if self.name in OPERATORS:
        return f"{self.name}({', '.join(child.to_readable_string(var_map) for child in self.children)})"
    else:
        return var_map.get(self.name, str(self.name))

Node.to_readable_string = to_readable_string

In [ ]:
def generate_random_tree(max_depth, feature_count, current_depth=0, parent_op=None):
    # Make deeper trees more likely
    terminal_prob = 0.1 + 0.4 * (current_depth / max_depth) ## (more terminals → smaller trees)

    if current_depth >= max_depth or random.random() < terminal_prob:
        # Prefer features (70%) over constants (30%)
        if random.random() < 0.7:
            return Node(f'x{random.randint(0, feature_count - 1)}')
        else:
            return Node(str(round(random.uniform(-5, 5), 2)))
    else:
        # Add operator variety
        op_weights = {
            '+': 0.18, '-': 0.18, '*': 0.20, '/': 0.07,
            'sin': 0.06, 'cos': 0.06,
            'log': 0.06, 'exp': 0.04,
            'square': 0.06, 'cube': 0.03,
            'pow': 0.06
        }

        op = random.choices(list(op_weights.keys()), weights=list(op_weights.values()), k=1)[0]
        func, arity = OPERATORS[op]
        children = [generate_random_tree(max_depth, feature_count, current_depth + 1, op) for _ in range(arity)]
        return Node(op, children)


In [ ]:
# Recursively collect all nodes along with their parent nodes
def collect_nodes_with_parents(node, parent=None):
    nodes = [(node, parent)]
    for child in node.children:
        nodes.extend(collect_nodes_with_parents(child, node))
    return nodes

## Crossover
def crossover(tree1, tree2):
    child1 = copy.deepcopy(tree1)
    child2 = copy.deepcopy(tree2)

    # Collect all nodes with their parent references
    nodes1 = collect_nodes_with_parents(child1)
    nodes2 = collect_nodes_with_parents(child2)

    # Prefer internal nodes (i.e., operators), fallback to any node
    op_nodes1 = [n for n in nodes1 if n[0].name in OPERATORS]
    op_nodes2 = [n for n in nodes2 if n[0].name in OPERATORS]

    if op_nodes1 and op_nodes2:
        (target1, parent1), (target2, parent2) = random.choice(op_nodes1), random.choice(op_nodes2)
    else:
        (target1, parent1), (target2, parent2) = random.choice(nodes1), random.choice(nodes2)

    # Swap the subtrees
    if parent1 is None:
        child1 = target2
    else:
        idx = parent1.children.index(target1)
        parent1.children[idx] = target2

    if parent2 is None:
        child2 = target1
    else:
        idx = parent2.children.index(target2)
        parent2.children[idx] = target1

    # print crossover to check
    # print(f"[CROSSOVER] Parent1: {tree1.to_string()}")
    # print(f"[CROSSOVER] Parent2: {tree2.to_string()}")
    # print(f"[CROSSOVER] Child1 : {child1.to_string()}")
    # print(f"[CROSSOVER] Child2 : {child2.to_string()}")

    return child1, child2


# # # Mutation
def mutate(tree, max_depth, feature_count):
    tree_copy = copy.deepcopy(tree)
    #tree_copy = tree.copy()
    nodes = collect_nodes_with_parents(tree_copy)
    target_node, parent = random.choice(nodes)

    mutation_type = random.choice(['subtree', 'operator', 'constant', 'feature'])

    if mutation_type == 'subtree':
        new_subtree = generate_random_tree(max_depth=max_depth, feature_count=feature_count)
        if parent is None:
            return new_subtree
        else:
            idx = parent.children.index(target_node)
            parent.children[idx] = new_subtree

    elif mutation_type == 'operator' and target_node.name in OPERATORS:
        current_op = target_node.name
        valid_ops = [op for op in OPERATORS if OPERATORS[op][1] == OPERATORS[current_op][1] and op != current_op]
        if valid_ops:
            new_op = random.choice(valid_ops)
            target_node.name = new_op

    elif mutation_type == 'constant' and is_constant(target_node.name):
        new_val = round(random.uniform(-10, 10), 2)
        target_node.name = new_val

    elif mutation_type == 'feature' and isinstance(target_node.name, str) and target_node.name.startswith('x'):
        old_feature = target_node.name
        new_feature = old_feature
        while new_feature == old_feature:
            new_feature = f'x{random.randint(0, feature_count - 1)}'
        target_node.name = new_feature
        #print(f"[MUTATION] Replaced feature {old_feature} → {new_feature}")

    return tree_copy



In [ ]:
class Individual:
    def __init__(self, tree, X, y, feature_names, validation_labels=None):
        self.tree = tree
        self.X = X
        self.y = y
        self.feature_names = feature_names
        self.validation_labels = validation_labels

        self.rank = None
        self.crowding_distance = 0.0

        self._evaluate()
        #print(f"[Debug] Feature names: {self.feature_names}")


    def _evaluate(self):
        try:
            # Evaluate predictions
            y_pred = np.clip([self.tree.evaluate(row) for row in self.X], -1e3, 1e3)
            self.r2 = r2_score(self.y, y_pred)
            self.predicted = y_pred
        except Exception:
            self.r2 = -np.inf
            self.predicted = [None] * len(self.y)

        # complexity calculation
        #self.complexity = self.tree.count_nodes()
        self.complexity = self.tree.weighted_complexity()

        # Spearman rank correlation (if validation labels exist)
        if self.validation_labels is not None and self.predicted is not None:
            try:
                predictions = np.array(self.predicted)
                dfq = pd.DataFrame({'predictions': predictions})
                unique_values = np.unique(predictions)

                if len(unique_values) > 5:
                    try:
                        dfq['predicted_class'] = pd.cut(dfq['predictions'], 5, labels=[1, 2, 3, 4, 5])
                    except ValueError:
                        dfq['predicted_class'] = 3  # fallback
                else:
                    dfq['predicted_class'] = 3  # fallback

                dfq['target_class'] = self.validation_labels

                predicted_classes = dfq['predicted_class'].to_numpy()
                target_classes = dfq['target_class'].to_numpy()

                if np.all(predicted_classes == predicted_classes[0]):
                    self.spearman_corr, self.spearman_p = 0.0, 1.0
                else:
                    corr, pval = spearmanr(predicted_classes, target_classes)
                    self.spearman_corr = 0.0 if np.isnan(corr) else corr
                    self.spearman_p = pval

            except Exception as e:
                print(f"[Spearman Error] {e}")
                self.spearman_corr = None
                self.spearman_p = None
        else:
            self.spearman_corr = None
            self.spearman_p = None


    @property
    def fitness(self):
        return [-self.r2, self.complexity] #-r2= maximize r2

    @property
    def symbolic(self):
        return str(self.tree)

    @property
    def readable(self):
        return self.tree.to_readable_string(self.feature_names)


    def copy(self):
        return Individual(copy.deepcopy(self.tree), self.X, self.y, self.feature_names, self.validation_labels)

    # def score(self, alpha=1.0, beta=0.01):
    #     return alpha * self.r2 - beta * self.complexity


In [ ]:
# Check if one individual dominates another
def dominates(ind_a, ind_b):
    f1_a, f2_a = ind_a.fitness
    f1_b, f2_b = ind_b.fitness
    return (f1_a <= f1_b and f2_a <= f2_b) and (f1_a < f1_b or f2_a < f2_b)

# Non-dominated sort to identify Pareto fronts
def non_dominated_sort(population):
    fronts = []
    dominated_count = {}
    dominates_list = {}
    individuals = list(range(len(population)))

    for p in individuals:
        dominated_count[p] = 0
        dominates_list[p] = []
        for q in individuals:
            if p == q:
                continue
            if dominates(population[p], population[q]):
                dominates_list[p].append(q)
            elif dominates(population[q], population[p]):
                dominated_count[p] += 1
        if dominated_count[p] == 0:
            if len(fronts) == 0:
                fronts.append([])
            fronts[0].append(p)

    i = 0
    while i < len(fronts) and len(fronts[i]) > 0:
        next_front = []
        for p in fronts[i]:
            for q in dominates_list[p]:
                dominated_count[q] -= 1
                if dominated_count[q] == 0:
                    next_front.append(q)
        i += 1
        if next_front:
            fronts.append(next_front)

    return fronts


In [ ]:
# NSGA-II survivor selection using non-dominated sorting and crowding distance

def objective_value(ind, obj):
    if obj == 'r2':
        return -ind.r2   # convert R² to minimization
    elif obj == 'complexity':
        return ind.complexity
    else:
        return getattr(ind, obj)


def compute_crowding_distance(front, objectives):
    size = len(front)
    if size == 0:
        return {}

    distance = {ind: 0.0 for ind in front}

    for obj in objectives:
        sorted_front = sorted(front, key=lambda ind: objective_value(ind, obj))

        distance[sorted_front[0]] = float('inf')
        distance[sorted_front[-1]] = float('inf')

        obj_min = objective_value(sorted_front[0], obj)
        obj_max = objective_value(sorted_front[-1], obj)

        if obj_max == obj_min:
            continue

        for j in range(1, size - 1):
            prev_val = objective_value(sorted_front[j - 1], obj)
            next_val = objective_value(sorted_front[j + 1], obj)

            distance[sorted_front[j]] += (next_val - prev_val) / (obj_max - obj_min)

    return distance

def select_survivors(population, num_survivors):
    fronts = non_dominated_sort(population)
    survivors = []
    objectives = ['r2', 'complexity']  # used for crowding distance

    for front_indices in fronts:
        front = [population[i] for i in front_indices]
        if len(survivors) + len(front) <= num_survivors:
            survivors.extend(front)
        else:
            distances = compute_crowding_distance(front, objectives)
            sorted_front = sorted(front, key=lambda ind: -distances[ind])
            survivors.extend(sorted_front[:num_survivors - len(survivors)])
            break

    return survivors



## Assign NSGA-II rank and crowding_distance to individuals in `population`

def assign_rank_and_crowding(population, objectives=('r2', 'complexity')):

    fronts = non_dominated_sort(population)

    # set defaults
    for ind in population:
        ind.rank = float('inf')
        ind.crowding_distance = 0.0

    for rank, front_indices in enumerate(fronts):
        front = [population[i] for i in front_indices]

        # assign rank
        for ind in front:
            ind.rank = rank

        # assign crowding distance
        distances = compute_crowding_distance(front, list(objectives))
        for ind in front:
            ind.crowding_distance = distances.get(ind, 0.0)


## lower rank wins and if rank tie -> higher crowding_distance wins

def crowded_tournament_selection(population, k):
    selected = []
    for _ in range(k):
        a, b = random.sample(population, 2)

        if a.rank < b.rank:
            selected.append(a)
        elif b.rank < a.rank:
            selected.append(b)
        else:
            # same rank -> prefer higher crowding distance
            if a.crowding_distance > b.crowding_distance:
                selected.append(a)
            elif b.crowding_distance > a.crowding_distance:
                selected.append(b)
            else:
                selected.append(a if random.random() < 0.5 else b)

    return selected

In [ ]:
def generate_offspring(parents, num_offspring, max_depth, feature_count, mutation_rate=0.3, crossover_rate=1.0, verbose=True):
    offspring = []
    while len(offspring) < num_offspring:
        p1, p2 = random.sample(parents, 2)

        # Apply crossover according to crossover probability
        if crossover_rate >= 1.0 or random.random() < crossover_rate:
            child1_tree, child2_tree = crossover(p1.tree, p2.tree)
        else:
            child1_tree = copy.deepcopy(p1.tree)
            child2_tree = copy.deepcopy(p2.tree)

        child1 = Individual(child1_tree, p1.X, p1.y, p1.feature_names)
        child2 = Individual(child2_tree, p2.X, p2.y, p2.feature_names)


        # Mutation for Child1
        if random.random() < mutation_rate:
            if random.random() < 0.05:  # only 5% chance of full reset
                new_tree = generate_random_tree(max_depth, feature_count)

                if verbose:
                    print(f"[MUTATION] Child1 replaced by new random tree: {new_tree.to_string()}")

                child1 = Individual(new_tree, p1.X, p1.y, p1.feature_names)

            else:
                if verbose:
                    print(f"[MUTATION] Before mutation (Child1): {child1.tree.to_string()}")

                mutated_tree = mutate(child1.tree, max_depth, feature_count)

                if verbose:
                    print(f"[MUTATION] After mutation  (Child1): {mutated_tree.to_string()}")

                child1 = Individual(mutated_tree, p1.X, p1.y, p1.feature_names)

        # Mutation for Child2
        if random.random() < mutation_rate:
            if random.random() < 0.05:
                new_tree = generate_random_tree(max_depth, feature_count)
                if verbose:
                    print(f"[MUTATION] Child2 replaced by new random tree: {new_tree.to_string()}")
                child2 = Individual(new_tree, p2.X, p2.y, p2.feature_names)
            else:
                if verbose:
                    print(f"[MUTATION] Before mutation (Child2): {child2.tree.to_string()}")
                mutated_tree = mutate(child2.tree, max_depth, feature_count)
                if verbose:
                    print(f"[MUTATION] After mutation  (Child2): {mutated_tree.to_string()}")
                child2 = Individual(mutated_tree, p2.X, p2.y, p2.feature_names)

        offspring.append(child1)
        if len(offspring) < num_offspring:
            offspring.append(child2)

    return offspring[:num_offspring]


In [ ]:
## pareto front method

def get_pareto_front(population):
    """Pareto front using fitness = [-r2, complexity]."""
    pareto = []

    for i, ind in enumerate(population):
        dominated = False

        for j, other in enumerate(population):
            if i == j:
                continue

            if (
                other.fitness[0] <= ind.fitness[0] and
                other.fitness[1] <= ind.fitness[1] and
                (
                    other.fitness[0] < ind.fitness[0] or
                    other.fitness[1] < ind.fitness[1]
                )
            ):
                dominated = True
                break

        if not dominated:
            pareto.append(ind)

    return pareto

###TEST #####.   feature track per generation
from collections import Counter

feature_usage_log = []  # Global tracker


def count_features(tree):
    if isinstance(tree.name, str) and tree.name.startswith('x'):
        return Counter([tree.name])
    counts = Counter()
    for child in tree.children:
        counts += count_features(child)
    return counts


In [ ]:
all_logs = []


In [ ]:
def create_individual_population(n_individuals, max_depth, feature_count, X, y, feature_names):
    population = []
    for _ in range(n_individuals):
        tree = generate_random_tree(max_depth=max_depth, feature_count=feature_count)
        ind = Individual(tree, X, y, feature_names)
        population.append(ind)
    return population

#all_logs = []

def run_gp_evolution(
    X, y, feature_names,
    population_size=100, #default value
    generations=100,  #default value
    max_depth=6,  #default value
    mutation_rate=.3,  #default value
    crossover_rate=1.0,
    log_file_path=None,
    collect_detailed_logs=True
):
    #Initial population
    population = create_individual_population(
        n_individuals=population_size,
        max_depth=max_depth,
        feature_count=len(feature_names),
        X=X,
        y=y,
        feature_names=feature_names
    )

    global_archive_map = {}

    gen_expr_log = None

    if collect_detailed_logs:
        if log_file_path is None:
            gen_expr_log_path = '/content/drive/My Drive/GWL/r2_complexity/train219_val99_2obj of GP/train219_all_generation_expressions_log.txt'
        else:
            gen_expr_log_path = log_file_path

        gen_expr_log = open(gen_expr_log_path, 'w')


    #feature_usage_log = []  # TEST global tracker for features track in all generations
    generation_pareto_fronts = []
    for gen in range(generations):
        #print(f"\nGeneration {gen + 1}/{generations}")

        assign_rank_and_crowding(population, objectives=('r2', 'complexity'))

        parents = crowded_tournament_selection(population, k=population_size)

        offspring = generate_offspring(
            parents=parents,
            num_offspring=population_size,
            max_depth=max_depth,
            feature_count=len(feature_names),
            mutation_rate=mutation_rate,
            crossover_rate=crossover_rate,
            verbose=collect_detailed_logs
        )

        combined_population = population + offspring

        population = select_survivors(
            combined_population,
            num_survivors=population_size
        )



        gen_front = get_pareto_front(population)

        generation_pareto_fronts.append([
            (ind.r2, ind.complexity) for ind in gen_front
        ])

        for ind in gen_front:
            key = ind.symbolic
            if key not in global_archive_map or ind.r2 > global_archive_map[key].r2:
                global_archive_map[key] = ind


        if collect_detailed_logs:
            gen_header = f"\nGeneration {gen + 1}"
            print(gen_header)
            gen_expr_log.write(gen_header + '\n')

            for i, ind in enumerate(population):
                expr_line = f"Individual {i + 1}, R²: {ind.r2:.4f}, Complexity: {ind.complexity:.4f}, Expr: {ind.readable}"
                print(expr_line)
                gen_expr_log.write(expr_line + '\n')

        ### test features over generation
        # Collect feature usage for this generation
        if collect_detailed_logs:
            generation_usage = Counter()
            for ind in population:
                generation_usage += count_features(ind.tree)
            feature_usage_log.append(generation_usage)


### for all log in excel

        if collect_detailed_logs:
            for i, ind in enumerate(population):
                try:
                    y_pred = [ind.tree.evaluate(row) for row in ind.X]
                except Exception:
                    y_pred = [None] * len(ind.y)

                target_str = ",".join([f"{v:.4f}" for v in ind.y])
                pred_str = ",".join([
                    f"{v:.4f}" if v is not None else "NaN"
                    for v in y_pred
                ])

                all_logs.append({
                    "Generation": gen + 1,
                    "Individual": i,
                    "R2": ind.r2,
                    "Complexity": ind.complexity,
                    "Targets": target_str,
                    "Predictions": pred_str,
                })


    # Global Pareto front from all generations
    global_pareto = get_pareto_front(list(global_archive_map.values()))

    print("\n--- All-Generations Global Pareto Front ---")
    for ind in global_pareto:
        print(f"R²: {ind.r2:.4f}, Complexity: {ind.complexity:.4f}, Expr: {ind.readable}")

    if gen_expr_log is not None:
        gen_expr_log.close()

    return population, generation_pareto_fronts  # Final evolved population
    #return population, feature_usage_log  # Adding feature log



In [ ]:
# =========================
# MULTI-RUN WITH CLEAN PRINTING
# =========================

NUM_RUNS = 20  #20
BASE_SEED = 42
SEEDS = [BASE_SEED + i for i in range(NUM_RUNS)]

ALL_PARETO_TXT = "/content/drive/My Drive/GWL/r2_complexity/train219_val99_2obj of GP/MultiObj_all_runs_pareto_fronts.txt"

# Clear file before starting
open(ALL_PARETO_TXT, "w").close()

all_run_pareto_fronts = []
all_run_generation_pareto_fronts = []

for run_idx, seed in enumerate(SEEDS, start=1):


    print(f"\n RUN {run_idx}\n")


    random.seed(seed)
    np.random.seed(seed)

    # reset logs if you use them
    all_logs = []

    # ---- Evolution ----
    final_pop, generation_pareto_fronts = run_gp_evolution(
        X=X,
        y=y,
        feature_names=symbolic_variable_map,
        population_size=100,   #150 #120 #100
        generations=100,     #100
        max_depth=6,
        mutation_rate=0.3,
        crossover_rate=1.0
    )
    all_run_generation_pareto_fronts.append(generation_pareto_fronts)

    # ---- Pareto ----
    pareto_front = get_pareto_front(final_pop)
    if len(pareto_front) == 0:
        print(f"WARNING: Empty Pareto front for run {run_idx}")
    print(f"\n[SUMMARY] Run {run_idx}: pareto_size={len(pareto_front)} | best_R2={max(ind.r2 for ind in pareto_front):.4f}")
    all_run_pareto_fronts.append(pareto_front)


    print(f"\n FINAL PARETO FRONT — RUN {run_idx} (size={len(pareto_front)})")

    # Print to console
    for ind in pareto_front:
        print(f"R2: {ind.r2:.6f} | Complexity: {ind.complexity:.6f} | Expr: {ind.readable}")

    # Print in single TXT file
    with open(ALL_PARETO_TXT, "a") as f:
        f.write(f"\nPARETO FRONT FOR RUN {run_idx} (size={len(pareto_front)})\n")
        # Write summary line once
        best_r2 = max(ind.r2 for ind in pareto_front)
        f.write(f"Best R2: {best_r2:.4f}\n")
        # Then write each individual
        for ind in pareto_front:
            f.write(f"R2: {ind.r2:.6f} | Complexity: {ind.complexity:.6f} | Expr: {ind.readable}\n")

print("\nAll runs completed.")
print(f"All Pareto fronts saved in: {ALL_PARETO_TXT}")

In [ ]:
## SAVE ALL (R2, Complexity) over generation for all run --- for HV convergence

import pickle

folder = "/content/drive/My Drive/GWL/r2_complexity/train219_val99_2obj of GP/"

# Save the data for later calculations
with open(folder + "BioSym_generation_pareto_fronts_20runs_for_convergence.pkl", "wb") as f:
    pickle.dump(all_run_generation_pareto_fronts, f)

# Save a readable text version
with open(folder + "BioSym_generation_pareto_fronts_20runs_for_convergence.txt", "w") as f:
    for run_number, run_fronts in enumerate(
        all_run_generation_pareto_fronts, start=1
    ):
        f.write(f"\n{'=' * 50}\nRUN {run_number}\n{'=' * 50}\n")

        for generation_number, front in enumerate(run_fronts, start=1):
            f.write(f"\nGeneration {generation_number}\n")
            f.write("R2\tComplexity\n")

            for r2, complexity in front:
                f.write(f"{r2:.10f}\t{complexity:.10f}\n")

print("Both .pkl and .txt files saved successfully.")

In [ ]:
#CHECK the Pickel file --- for HV convergence

import pickle

file_path = (
    "/content/drive/My Drive/GWL/r2_complexity/"
    "train219_val99_2obj of GP/"
    "BioSym_generation_pareto_fronts_20runs_for_convergence.pkl"
)

with open(file_path, "rb") as f:
    saved_fronts = pickle.load(f)

print("Runs:", len(saved_fronts))
print("Generations per run:", [len(run) for run in saved_fronts])
print("Total generation-wise fronts:", sum(len(run) for run in saved_fronts))

In [ ]:
#min-max --- for HV convergence

import numpy as np

# Collect objective values from all runs and all generations
all_points = np.array([
    (1.0 - r2, complexity)
    for run_fronts in saved_fronts
    for front in run_fronts
    for r2, complexity in front
], dtype=float)

# One common normalization range for the entire convergence analysis
objective_min = all_points.min(axis=0)
objective_max = all_points.max(axis=0)

print("Number of stored points:", len(all_points))
print("Minimum (1-R2, Complexity):", objective_min)
print("Maximum (1-R2, Complexity):", objective_max)

In [ ]:
##HV calculation

import numpy as np

reference_point = np.array([1.1, 1.1], dtype=float)

def calculate_hv(front):
    # Convert to the two minimization objectives
    points = np.array(
        [(1.0 - r2, complexity) for r2, complexity in front],
        dtype=float
    )

    # Apply the SAME normalization to every front
    normalized = (points - objective_min) / (objective_max - objective_min)

    # Remove duplicate objective points
    normalized = np.unique(normalized, axis=0)

    # Sort by the first objective (ascending)
    normalized = normalized[np.argsort(normalized[:, 0])]

    # Calculate two-dimensional hypervolume
    hv = 0.0
    current_y = reference_point[1]

    for x, y in normalized:
        if x >= reference_point[0] or y >= reference_point[1]:
            continue

        if y < current_y:
            hv += (reference_point[0] - x) * (current_y - y)
            current_y = y

    return hv


# Calculate one HV value per generation, per run
all_run_hv = np.array([
    [calculate_hv(front) for front in run_fronts]
    for run_fronts in saved_fronts
])

print("HV array shape:", all_run_hv.shape)
print("Mean HV at generation 1:", all_run_hv[:, 0].mean())
print("Mean HV at generation 100:", all_run_hv[:, -1].mean())

In [ ]:
# =====================================================
# BioSym: ## Plot HV for convergence
# =====================================================

import os
import numpy as np
import matplotlib.pyplot as plt

# Generations
generations = np.arange(1, 101)

# Mean HV across 20 runs at each generation
mean_hv = all_run_hv.mean(axis=0)
std_hv = all_run_hv.std(axis=0, ddof=1)

# Create output folder if needed
pdf_dir = "/content/drive/My Drive/GWL/r2_complexity/train219_val99_2obj of GP/Figure_pdf"
os.makedirs(pdf_dir, exist_ok=True)

# Journal-style font settings
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 10,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})

# Create figure
fig, ax = plt.subplots(figsize=(6.2, 4.4), facecolor="white")

# Plot mean HV
ax.plot(
    generations,
    mean_hv,
    linewidth=2.0
)

# Axis labels
ax.set_xlabel("Generation", labelpad=6)
ax.set_ylabel("Mean hypervolume (HV)", labelpad=6)

# Axis limits and ticks
ax.set_xlim(1, 100)
ax.set_xticks([1, 20, 40, 60, 80, 100])

# Light grid
ax.grid(True, linewidth=0.6, alpha=0.25)

# Tight layout
plt.tight_layout()

# Save as PDF
pdf_file = os.path.join(pdf_dir, "BioSym_convergence_HV.pdf")

fig.savefig(
    pdf_file,
    format="pdf",
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close(fig)

print("convergence figure saved successfully!")
print("File:", pdf_file)

In [ ]:
# =====================================================
# BioSym: ## Table HV for convergence
# =====================================================

import pandas as pd

convergence_table = pd.DataFrame({
    "Generation": generations,
    "Mean_HV": mean_hv,
    "SD_HV": std_hv
})

output_path = (
    "/content/drive/My Drive/GWL/r2_complexity/"
    "train219_val99_2obj of GP/"
    "BioSym_convergence_HV_20runs.csv"
)

convergence_table.to_csv(output_path, index=False)

print(convergence_table.iloc[[0, 19, 39, 59, 79, 99]].to_string(index=False))
print("\nSaved to:", output_path)

In [ ]:
# =========================
# pooled GLOBAL PARETO FRONT (Front 0 of all runs)---- non-dominated sorting again
# =========================

# Pool all solutions
pooled = []
for pf in all_run_pareto_fronts:
    pooled.extend(pf)

print("Total pooled before duplicate removal:", len(pooled))

# Remove duplicates based on (R2, Complexity)
unique_dict = {}
for ind in pooled:
    key = (round(ind.r2, 10), round(ind.complexity, 10))
    if key not in unique_dict:
        unique_dict[key] = ind

unique_solutions = list(unique_dict.values())

print("After duplicate removal:", len(unique_solutions))

# Compute global Pareto
global_pareto_front = get_pareto_front(unique_solutions) ## non-dominated sorting again

print("Global Pareto size:", len(global_pareto_front))

# save global pareto to txt
GLOBAL_TXT = "/content/drive/My Drive/GWL/r2_complexity/train219_val99_2obj of GP/MultiObj_global_pareto_front.txt"

# Sort by R2 descending
global_pareto_front.sort(key=lambda ind: ind.r2, reverse=True)


print(f"\nGLOBAL PARETO FRONT (Front 0 of all runs - remove duplicates) | size: {len(global_pareto_front)}")
print("="*70)

for ind in global_pareto_front:
    print(f"R2: {ind.r2:.6f} | Complexity: {ind.complexity:.6f} | Expr: {ind.readable}")

with open(GLOBAL_TXT, "w") as f:
    f.write(f"GLOBAL PARETO FRONT (Front 0 of all runs-remove duplicates) | size: {len(global_pareto_front)} \n\n")

    for ind in global_pareto_front:
        f.write(f"R2: {ind.r2:.6f} | Complexity: {ind.complexity:.6f} | Expr: {ind.readable}\n")

print("\nSaved global pareto to:", GLOBAL_TXT)

print(f"\nR2 range: [{min(ind.r2 for ind in global_pareto_front):.4f}, {max(ind.r2 for ind in global_pareto_front):.4f}]")
print(f"Complexity range: [{min(ind.complexity for ind in global_pareto_front)}, {max(ind.complexity for ind in global_pareto_front)}]")

In [ ]:
# ==========================================================
# STEP 2A: LOAD AND CHECK THE VALIDATION DATASET
# ==========================================================

validation_file_path = "/content/drive/My Drive/GWL/r2_complexity/train219_val99_2obj of GP/validation_test99_dataset.csv"

validation_df = pd.read_csv(validation_file_path)

print("Validation dataset shape:", validation_df.shape)
print("\nValidation columns:")
print(validation_df.columns.tolist())

# ==========================================================
# STEP 2B: PREPARE VALIDATION INPUTS AND TARGET
# ==========================================================

feature_columns = [
    'Soil',
    'Slope',
    'Rain',
    'LULC',
    'Lithology',
    'Lineament density',
    'Geomorphology',
    'Drainage density'
]

target_column = 'Net Groundwater Recharge'

X_val = validation_df[feature_columns].copy()
y_val = validation_df[target_column].to_numpy(dtype=float)

print("Validation input shape:", X_val.shape)
print("Validation target shape:", y_val.shape)

print("\nMissing values in predictors:")
print(X_val.isnull().sum())

print("\nMissing values in target:", validation_df[target_column].isnull().sum())

In [ ]:
# ==========================================================
# STEP 2C: EVALUATE GLOBAL PARETO EXPRESSIONS ON VALIDATION DATA
# ==========================================================

X_val_array = X_val.to_numpy(dtype=float)

validation_results = []

for i, ind in enumerate(global_pareto_front, start=1):

    # Evaluate expression on validation data
    y_pred_val = np.clip(
        [ind.tree.evaluate(row) for row in X_val_array],
        -1e3,
        1e3
    )

    # Calculate validation R²
    val_r2 = r2_score(y_val, y_pred_val)

    # Calculate expression complexity
    complexity = ind.tree.weighted_complexity()

    # Store results
    validation_results.append({
        "Solution": f"P{i}",
        "Training_R2": ind.r2,
        "Validation_R2": val_r2,
        "Complexity": complexity,
        "Expression": ind.readable
    })


# ==========================================================
# PRINT RESULTS AND SAVE THE SAME RESULTS TO TXT FILE
# ==========================================================

output_file = "/content/drive/My Drive/GWL/r2_complexity/train219_val99_2obj of GP/global_pareto_validation_r2.txt"

lines = []

for result in validation_results:

    block = (
        f"Solution      : {result['Solution']}\n"
        f"Training R²   : {result['Training_R2']:.6f}\n"
        f"Validation R² : {result['Validation_R2']:.6f}\n"
        f"Complexity    : {result['Complexity']:.6f}\n"
        f"Expression    : {result['Expression']}\n"
        + "-" * 100
    )

    lines.append(block)

result_text = "\n".join(lines)

# Print once
print(result_text)

# Save exactly the same information to Google Drive
with open(output_file, "w", encoding="utf-8") as f:
    f.write(result_text)

print(f"\nResults saved to: {output_file}")

In [ ]:
# ==========================================================
# Training vs Validation R² for Pareto-optimal solutions
# ==========================================================

import numpy as np
import matplotlib.pyplot as plt
import os

# Extract training and validation R² dynamically
training_r2 = [result["Training_R2"] for result in validation_results]
validation_r2 = [result["Validation_R2"] for result in validation_results]

# Check that training and validation results have the same length
if len(training_r2) != len(validation_r2):
    raise ValueError("Training and validation R² lists must have the same length.")

# Number of Pareto solutions determined automatically
solutions = np.arange(1, len(training_r2) + 1)
x = np.arange(len(solutions))

# Output path
figure_path = (
    "/content/drive/My Drive/GWL/r2_complexity/"
    "train219_val99_2obj of GP/Figure_pdf/"
    "training_validation_pareto_r2.pdf"
)

os.makedirs(os.path.dirname(figure_path), exist_ok=True)


plt.rcParams.update({
    "font.family": "serif",
    "font.size": 10,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})

# Create figure
fig, ax = plt.subplots(figsize=(7.0, 4.8), facecolor="white")

# Width of grouped bars
bar_width = 0.38

# Training R² bars
ax.bar(
    x - bar_width / 2,
    training_r2,
    width=bar_width,
    label=r"Training $R^2$"
)

# Validation R² bars
ax.bar(
    x + bar_width / 2,
    validation_r2,
    width=bar_width,
    label=r"Validation $R^2$"
)

# Reference line at R² = 0
ax.axhline(
    0,
    linestyle="--",
    linewidth=0.9
)

# Axis labels
ax.set_xlabel("Pareto-optimal solution", labelpad=6)
ax.set_ylabel(r"$R^2$", labelpad=6)

# Dynamic solution labels
ax.set_xticks(x)
ax.set_xticklabels([f"S{i}" for i in solutions])

# Light horizontal grid
ax.grid(axis="y", linewidth=0.6, alpha=0.25)

# Legend
ax.legend(frameon=False, loc="upper right")

# Optional: make x-limits slightly cleaner
ax.set_xlim(-0.5, len(x) - 0.5)

# Tight layout
fig.tight_layout()

# Save as PDF
fig.savefig(
    figure_path,
    format="pdf",
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close(fig)

print(f"Journal-style figure saved to: {figure_path}")
print(f"File size: {os.path.getsize(figure_path)} bytes")

In [ ]:
# ==========================================================
# OBSERVE PLOTS + SAVE VALIDATION DETAILS TO EXCEL
# ==========================================================

import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

excel_path = "/content/drive/My Drive/GWL/r2_complexity/train219_val99_2obj of GP/global_pareto_validation_results.xlsx"
os.makedirs(os.path.dirname(excel_path), exist_ok=True)

# ----------------------------------------------------------
# Dynamic plot layout
# ----------------------------------------------------------
n_solutions = len(global_pareto_front)

n_cols = 4
n_rows = math.ceil(n_solutions / n_cols)

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(16, 4 * n_rows)
)

axes = np.atleast_1d(axes).flatten()

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:

    for i, ind in enumerate(global_pareto_front):

        # Validation predictions
        y_pred_val = np.clip(
            [ind.tree.evaluate(row) for row in X_val_array],
            -1e3, 1e3
        )

        val_r2 = r2_score(y_val, y_pred_val)
        complexity = ind.tree.weighted_complexity()

        # --------------------------------------------------
        # Plot
        # --------------------------------------------------
        ax = axes[i]

        ax.scatter(y_val, y_pred_val, alpha=0.7, s=20)

        vmin = min(y_val.min(), y_pred_val.min())
        vmax = max(y_val.max(), y_pred_val.max())

        ax.plot(
            [vmin, vmax],
            [vmin, vmax],
            linestyle='--',
            linewidth=1
        )

        ax.set_title(
            f'S{i+1}: Validation $R^2$ = {val_r2:.3f}',
            fontsize=10
        )

        ax.set_xlabel('Observed/Actual', fontsize=9)
        ax.set_ylabel('Estimated/Computed by Pareto', fontsize=9)
        ax.grid(alpha=0.2)

        # --------------------------------------------------
        # Save observations to Excel
        # --------------------------------------------------
        sheet_df = X_val.copy()
        sheet_df["Actual"] = y_val
        sheet_df["Predicted"] = y_pred_val

        sheet_df.to_excel(
            writer,
            sheet_name=f"S{i+1}",
            index=False,
            startrow=2
        )

        ws = writer.sheets[f"S{i+1}"]
        ws["A1"] = (
            f"Training R²: {ind.r2:.6f}, "
            f"Validation R²: {val_r2:.6f}, "
            f"Complexity: {complexity:.6f}, "
            f"Expression: {ind.readable}"
        )

# Hide unused subplot axes
for j in range(n_solutions, len(axes)):
    axes[j].set_visible(False)

fig.tight_layout()
plt.show()

print(f"Excel file saved to:\n{excel_path}")

In [ ]:
## Bi-objecttive

## Prefix to Infix
## Infix write in Latex (tex)

import re
import os

def prefix_to_infix(expr):
    """Convert prefix expression to infix notation"""
    expr = expr.strip()

    if ',' not in expr and '(' not in expr and ')' not in expr:
        return expr

    match = re.match(r'^([a-zA-Z+\-*/]+)\(', expr)
    if not match:
        return expr

    op = match.group(1)

    depth = 0
    content_start = len(op) + 1
    content_end = -1

    for i in range(content_start, len(expr)):
        if expr[i] == '(':
            depth += 1
        elif expr[i] == ')':
            if depth == 0:
                content_end = i
                break
            else:
                depth -= 1

    if content_end == -1:
        return expr

    content = expr[content_start:content_end]

    args = []
    current_arg = []
    depth = 0

    for char in content:
        if char == '(':
            depth += 1
            current_arg.append(char)
        elif char == ')':
            depth -= 1
            current_arg.append(char)
        elif char == ',' and depth == 0:
            args.append(''.join(current_arg).strip())
            current_arg = []
        else:
            current_arg.append(char)

    if current_arg:
        args.append(''.join(current_arg).strip())

    converted_args = [prefix_to_infix(arg) for arg in args]

    if op in ['+', '-', '*', '/']:
        if len(args) == 2:
            return f"({converted_args[0]} {op} {converted_args[1]})"
    elif op == 'log':
        return f"log({converted_args[0]})"
    elif op == 'sin':
        return f"sin({converted_args[0]})"
    elif op == 'square':
        return f"({converted_args[0]})^2"
    elif op == 'cube':
        return f"({converted_args[0]})^3"
    elif op == 'pow':
        return f"({converted_args[0]})^({converted_args[1]})"
    else:
        return f"{op}({', '.join(converted_args)})"

    return expr

def infix_to_latex(expr):
    """Convert infix expression to LaTeX format (paper safe)."""

    # Escape LaTeX special characters (feature names often have _)
    expr = expr.replace('&', r'\&')
    expr = expr.replace('%', r'\%')
    expr = expr.replace('_', r'\_')

    # Functions
    expr = expr.replace('sin(', r'\sin(')
    expr = expr.replace('cos(', r'\cos(')
    expr = expr.replace('log(', r'\log(')
    expr = expr.replace('exp(', r'\exp(')

    # Multiplication
    expr = expr.replace('*', r' * ')

    return expr

# File paths
file_path = r"/content/drive/My Drive/GWL/r2_complexity/train219_val99_2obj of GP/MultiObj_global_pareto_front.txt"
infix_output_path = r"/content/drive/My Drive/GWL/r2_complexity/train219_val99_2obj of GP/MultiObj_global_pareto_INFIX_format.txt"
latex_output_path = r"/content/drive/My Drive/GWL/r2_complexity/train219_val99_2obj of GP/MultiObj_global_pareto_latex.tex"

print("="*80)
print("CONVERTING ALL EXPRESSIONS TO INFIX")
print("="*80)

try:
    with open(file_path, 'r') as f:
        lines = f.readlines()

    # Store all expressions
    expressions = []

    print("\nPROCESSING EXPRESSIONS:")
    print("-"*60)

    for i, line in enumerate(lines, 1):
        line = line.strip()
        if line.startswith("R2:"):
            parts = line.split("|")
            r2 = parts[0].replace("R2:", "").strip()
            complexity = parts[1].replace("Complexity:", "").strip()
            expr = parts[2].replace("Expr:", "").strip()

            # Convert to infix
            infix = prefix_to_infix(expr)

            expressions.append({
                'r2': r2,
                'complexity': complexity,
                'prefix': expr,
                'infix': infix
            })

    # Print ALL expressions to console (PREFIX + INFIX)
    print(f"\n{'='*80}")
    print(f"ALL {len(expressions)} EXPRESSIONS - PREFIX vs INFIX")
    print(f"{'='*80}")

    for i, exp in enumerate(expressions, 1):
        print(f"\n{i:2d}. R²: {exp['r2']} | Complexity: {exp['complexity']}")
        print(f"   PREFIX: {exp['prefix']}")
        print(f"   INFIX : {exp['infix']}")
        print(f"   {'-'*50}")

    # Save ONLY INFIX to text file
    with open(infix_output_path, 'w') as f:
        f.write("="*80 + "\n")
        f.write(f"INFIX EXPRESSIONS FROM GLOBAL PARETO FRONT ({len(expressions)} equations)\n")
        f.write("="*80 + "\n\n")

        for i, exp in enumerate(expressions, 1):
            f.write(f"{i:2d}. R²: {exp['r2']} | Complexity: {exp['complexity']}\n")
            f.write(f"   INFIX EXPRESSION: {exp['infix']}\n\n")

    print(f"\n{'='*80}")
    print(f"✅ Saved INFIX ONLY to: {infix_output_path}")


    # ==========================================================
    # GENERATE LATEX TABLE WITH SOLUTION COLUMN
    # ==========================================================

    with open(latex_output_path, 'w', encoding='utf-8') as f:

        f.write(r"\begin{table}[htbp]" + "\n")
        f.write(r"\centering" + "\n")

        f.write(
            r"\caption{Pareto-optimal expressions obtained from BioSym}"
            + "\n"
        )

        f.write(r"\label{tab:pareto_front}" + "\n")
        f.write(r"\renewcommand{\arraystretch}{1.50}" + "\n")
        f.write(r"\small" + "\n")

        # Solution | R² | Complexity | Expression
        f.write(
            r"\begin{tabularx}{\textwidth}{"
            r">{\raggedright\arraybackslash}p{1.5cm} "
            r">{\raggedright\arraybackslash}p{1.5cm} "
            r">{\raggedright\arraybackslash}p{2.0cm} "
            r">{\raggedright\arraybackslash}X}"
            + "\n"
        )

        f.write(r"\hline" + "\n")

        f.write(
            r"\textbf{Solution} & \textbf{$R^2$} & "
            r"\textbf{Complexity} & \textbf{Expression} \\"
            + "\n"
        )

        f.write(r"\hline" + "\n")

        # Write all Pareto solutions
        for i, exp in enumerate(expressions, start=1):

            latex_expr = infix_to_latex(exp['infix'])

            r2_formatted = f"{float(exp['r2']):.6f}"
            complexity_formatted = f"{float(exp['complexity']):.2f}"

            f.write(
                f"S{i} & "
                f"{r2_formatted} & "
                f"{complexity_formatted} & "
                #f"${latex_expr}$ \\\\"
                f"${latex_expr}$ \\\\[3pt]"
                + "\n"
            )

        f.write(r"\hline" + "\n")
        f.write(r"\end{tabularx}" + "\n")
        f.write(r"\end{table}" + "\n")

    print(f"✅ Also saved LaTeX table to: {latex_output_path}")
    print("="*80)

except FileNotFoundError:
    print(f"❌ File not found: {file_path}")

except Exception as e:
    print(f"❌ Error: {e}")

In [ ]:
## PLOT GLOBAL PARETO

import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "axes.unicode_minus": False,

    "font.size": 10,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 8
})

#Plot 1: Complexity vs R²
def plot_global_pareto_r2_vs_complexity(global_pareto_front, title="Global Pareto (R² vs Complexity)"):
    x = [ind.complexity for ind in global_pareto_front]
    y = [ind.r2 for ind in global_pareto_front]

    #plt.figure(figsize=(7, 5))
    plt.figure(figsize=(3.3, 2.6))
    plt.scatter(x, y, s=25, edgecolors='black', linewidths=0.5)

    # Sort points for line
    # points = sorted(zip(x, y))
    # x_sorted, y_sorted = zip(*points)
    # plt.plot(x_sorted, y_sorted, linewidth=1.2)

    plt.xlabel("Complexity (minimize)")
    plt.ylabel("R² (maximize)")
    plt.title(title)
    #plt.grid(True)
    plt.grid(True, linestyle="--", linewidth=0.5, alpha=0.7)
    plt.tight_layout()
    #plt.savefig("/content/drive/My Drive/GWL/r2_complexity/train219_val99_2obj of GP/Figure_pdf/plot_r2_vs_complexity.pdf", bbox_inches='tight')
    plt.show()

plot_global_pareto_r2_vs_complexity(global_pareto_front)


#Plot 2: Complexity vs (1 − R²) (both minimize, bottom-left is best)
#def plot_global_pareto_1mr2_vs_complexity(global_pareto_front, title="Global Pareto ((1-R²) vs Complexity)"):
def plot_global_pareto_1mr2_vs_complexity(global_pareto_front):
    x = [ind.complexity for ind in global_pareto_front]
    y = [1.0 - ind.r2 for ind in global_pareto_front]

    #plt.figure(figsize=(7, 5))
    plt.figure(figsize=(3.3, 2.6))
    plt.scatter(x, y, s=25, edgecolors='black', linewidths=0.5)

    # Sort points for line
    # points = sorted(zip(x, y))
    # x_sorted, y_sorted = zip(*points)
    # plt.plot(x_sorted, y_sorted, linewidth=1.2)

    plt.xlabel("Complexity (minimize)")
    plt.ylabel("1 - R² (minimize)")
    #plt.title(title)
    #plt.grid(True)
    plt.grid(True, linestyle="--", linewidth=0.5, alpha=0.7)
    plt.tight_layout()
    plt.savefig("/content/drive/My Drive/GWL/r2_complexity/train219_val99_2obj of GP/Figure_pdf/plot_pareto_front.pdf", bbox_inches='tight')
    plt.show()

plot_global_pareto_1mr2_vs_complexity(global_pareto_front)



#Plot 3: R² vs 1/Complexity (both maximize top-right is best)
def plot_global_pareto_r2_vs_inv_complexity(global_pareto_front, title="Global Pareto (R² vs 1/Complexity)", eps=1e-12):
    x = [1.0 / (ind.complexity + eps) for ind in global_pareto_front]
    y = [ind.r2 for ind in global_pareto_front]

    #plt.figure(figsize=(7, 5))
    plt.figure(figsize=(3.3, 2.6))
    plt.scatter(x, y, s=25, edgecolors='black', linewidths=0.5)

    # Sort points for line
    # points = sorted(zip(x, y))
    # x_sorted, y_sorted = zip(*points)
    # plt.plot(x_sorted, y_sorted, linewidth=1.2)


    plt.xlabel("1 / Complexity (maximize)")
    plt.ylabel("R² (maximize)")
    plt.title(title)
    #plt.grid(True)
    plt.grid(True, linestyle="--", linewidth=0.5, alpha=0.7)
    plt.tight_layout()
    #plt.savefig("/content/drive/My Drive/GWL/r2_complexity/train219_val99_2obj of GP/Figure_pdf/plot_r2_vs_1_by_complexity.pdf", bbox_inches='tight')
    plt.show()

plot_global_pareto_r2_vs_inv_complexity(global_pareto_front)

In [ ]:
## Plot both bi-objective apareto front and single objective best expression

import matplotlib.pyplot as plt

def plot_global_pareto_1mr2_vs_complexity(
    global_pareto_front,
    best_complexity=None,
    best_r2=None,
    title="Global Pareto ((1-R²) vs Complexity)"
):
    x = [ind.complexity for ind in global_pareto_front]
    y = [1.0 - ind.r2 for ind in global_pareto_front]

    #plt.figure(figsize=(7, 5))
    plt.figure(figsize=(3.3, 2.6))

    # Pareto solutions (only scatter)
    plt.scatter(x, y, s=25, edgecolors='black', linewidths=0.5,
                label="Bi-objective")

    # Single-objective best point
    if best_complexity is not None and best_r2 is not None:
        plt.scatter(
            best_complexity,
            1 - best_r2,
            marker="*",
            s=200,
            edgecolors='black',
            label="Single-objective"
        )


    plt.xlabel("Complexity (minimize)")
    plt.ylabel("1 - R² (minimize)")
    #plt.title(title)
    plt.legend()
    plt.grid(True, linestyle="--", linewidth=0.5, alpha=0.7)

    plt.tight_layout()
    plt.savefig(
        "/content/drive/My Drive/GWL/r2_complexity/train219_val99_2obj of GP/Figure_pdf/plot_pareto_vs_single.pdf",
        bbox_inches='tight'
    )

    plt.show()

#call with 1624.70 → complexity, 0.692120 → R²
plot_global_pareto_1mr2_vs_complexity(
    global_pareto_front,
    best_complexity=1624.70,
    best_r2=0.692120
)

In [ ]:

# ==========================================================
#Pareto-front comparison----- Broken
# BioSym vs single-objective symbolic regression
# ==========================================================

import os
import matplotlib.pyplot as plt
import matplotlib as mpl

# Consistent journal-style formatting
mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "axes.unicode_minus": False,
    "font.size": 10,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})


def plot_pareto_vs_single_broken(
    global_pareto_front,
    best_complexity,
    best_r2
):
    # Use the original experimental results
    x_biosym = [ind.complexity for ind in global_pareto_front]
    y_biosym = [1.0 - ind.r2 for ind in global_pareto_front]

    x_single = best_complexity
    y_single = 1.0 - best_r2

    # Create two panels with a shared y-axis
    fig, (ax1, ax2) = plt.subplots(
        1, 2,
        sharey=True,
        figsize=(7.2, 4.4),
        facecolor="white",
        gridspec_kw={
            "width_ratios": [3.5, 1.3],
            "wspace": 0.06
        }
    )

    # Leave space for the legend and shared x-axis label
    fig.subplots_adjust(
        left=0.11,
        right=0.98,
        bottom=0.19,
        top=0.85
    )

    # BioSym Pareto solutions: left panel
    biosym_plot = ax1.scatter(
        x_biosym,
        y_biosym,
        s=48,
        marker="o",
        color="#1f77b4",
        edgecolors="black",
        linewidths=0.55,
        zorder=3,
        label="BioSym Pareto front"
    )

    # Single-objective solution: right panel
    single_plot = ax2.scatter(
        x_single,
        y_single,
        s=185,
        marker="*",
        color="#ff7f0e",
        edgecolors="black",
        linewidths=0.65,
        zorder=4,
        label="Single-objective"
    )

    # Preserve the broken complexity axis
    ax1.set_xlim(-1, 25)
    ax2.set_xlim(1590, 1660)

    ax1.set_xticks([0, 5, 10, 15, 20, 25])
    ax2.set_xticks([1600, 1625, 1650])

    # Shared objective axis
    ax1.set_ylim(0.20, 2.15)
    ax1.set_ylabel(r"$1 - R^2$ (minimize)", labelpad=6)

    fig.supxlabel(
        "Complexity (minimize)",
        x=0.54,
        y=0.06,
        fontsize=11
    )

    # Light grid behind the data
    for ax in (ax1, ax2):
        ax.set_axisbelow(True)
        ax.grid(
            axis="y",
            linestyle="--",
            linewidth=0.55,
            alpha=0.30
        )
        ax.tick_params(axis="both", length=3.5, width=0.8)

    # Hide the adjoining spines
    ax1.spines["right"].set_visible(False)
    ax2.spines["left"].set_visible(False)

    ax1.tick_params(axis="y", right=False)
    ax2.tick_params(
        axis="y",
        left=False,
        right=False,
        labelleft=False
    )

    # Broken-axis diagonal marks
    d = 0.012

    left_break_style = dict(
        transform=ax1.transAxes,
        color="black",
        clip_on=False,
        linewidth=1.0
    )

    ax1.plot(
        (1 - d, 1 + d),
        (1 - d, 1 + d),
        **left_break_style
    )
    ax1.plot(
        (1 - d, 1 + d),
        (-d, d),
        **left_break_style
    )

    right_break_style = dict(
        transform=ax2.transAxes,
        color="black",
        clip_on=False,
        linewidth=1.0
    )

    ax2.plot(
        (-d, d),
        (1 - d, 1 + d),
        **right_break_style
    )
    ax2.plot(
        (-d, d),
        (-d, d),
        **right_break_style
    )

    # Place legend above the plotting area
    # Legend inside the upper-right area of the figure
    fig.legend(
        handles=[biosym_plot, single_plot],
        labels=["BioSym Pareto front", "Single-objective"],
        loc="upper right",
        bbox_to_anchor=(0.96, 0.84),
        frameon=True,
        facecolor="white",
        edgecolor="0.8",
        fontsize=9
    )

    # Save the PDF for Overleaf
    output_file = (
        "/content/drive/My Drive/GWL/r2_complexity/"
        "train219_val99_2obj of GP/Figure_pdf/"
        "plot_pareto_vs_single_broken.pdf"
    )

    os.makedirs(os.path.dirname(output_file), exist_ok=True)

    fig.savefig(
        output_file,
        format="pdf",
        bbox_inches="tight",
        facecolor="white"
    )

    plt.show()
    plt.close(fig)

    print(f"Figure saved to: {output_file}")


# Original single-objective result
plot_pareto_vs_single_broken(
    global_pareto_front,
    best_complexity=1624.70,
    best_r2=0.692120
)

In [ ]:

# ==========================================================
# Journal-style Pearson correlation heatmap
# Training dataset: 219 samples
# ==========================================================

import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

# Google Drive should already be mounted in Colab

# Input and output paths
file_path = (
    "/content/drive/My Drive/GWL/r2_complexity/"
    "train219_val99_2obj of GP/train219_dataset.csv"
)

output_path = (
    "/content/drive/My Drive/GWL/r2_complexity/"
    "train219_val99_2obj of GP/Figure_pdf/"
    "heatmap_train219_with_target.pdf"
)

# Load the original training data
df = pd.read_csv(file_path)

columns = [
    "LULC",
    "Drainage density",
    "Geomorphology",
    "Soil",
    "Slope",
    "Rain",
    "Lineament density",
    "Lithology",
    "Net Groundwater Recharge"
]

# Calculate the original Pearson correlation values
corr_matrix = df[columns].corr(method="pearson")

# Mask the diagonal and upper triangle
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

# Display labels: line breaks make long labels easier to read
axis_labels = [
    "LULC",
    "Drainage\ndensity",
    "Geomorphology",
    "Soil",
    "Slope",
    "Rain",
    "Lineament\ndensity",
    "Lithology",
    "Net groundwater\nrecharge"
]

# Prepare annotation labels
# Display values that round to zero as 0.00, not -0.00.
# This changes only the displayed text, not the correlation values.
annotation_labels = corr_matrix.map(
    lambda value: "0.00" if abs(value) < 0.005 else f"{value:.2f}"
)


mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "axes.unicode_minus": False,
    "font.size": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})

fig, ax = plt.subplots(figsize=(8.2, 6.8), facecolor="white")

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=annotation_labels,
    fmt="",
    annot_kws={"size": 9},
    cmap="RdBu_r",
    vmin=-1,
    vmax=1,
    center=0,
    linewidths=0.45,
    linecolor="white",
    square=True,
    cbar=True,
    cbar_kws={
        "ticks": [-1, -0.5, 0, 0.5, 1],
        "shrink": 0.82,
        "pad": 0.025
    },
    ax=ax
)

# Axis labels
ax.set_xticklabels(
    axis_labels,
    rotation=50,
    ha="right",
    rotation_mode="anchor"
)

ax.set_yticklabels(
    axis_labels,
    rotation=0
)

# Remove unnecessary tick marks
ax.tick_params(axis="both", length=0, pad=5)

# Colorbar formatting
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=9, length=3)
#cbar.set_label("Pearson correlation (r)", fontsize=10, labelpad=8)

# Save PDF for Overleaf
os.makedirs(os.path.dirname(output_path), exist_ok=True)

fig.savefig(
    output_path,
    format="pdf",
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close(fig)

print(f"Figure saved to: {output_path}")

In [ ]:
## HV calculation + plotting together using (1 - R²) and complexity

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

# ---------------- Font / figure style ----------------
mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "axes.unicode_minus": False,
    "font.size": 10,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 8
})

# ---------------- HV reference point ----------------
HV_REF = (1.1, 1.1)

# ---------------- HV helper functions ----------------
def front_to_points_min(front):
    return np.array([[1.0 - ind.r2, ind.complexity] for ind in front], dtype=float)

def unique_points(points, decimals=10):
    pts = np.round(points, decimals=decimals)
    _, idx = np.unique(pts, axis=0, return_index=True)
    return points[np.sort(idx)]

def normalize_points(points, mins, maxs):
    denom = np.where((maxs - mins) == 0.0, 1.0, (maxs - mins))
    return (points - mins) / denom

def hypervolume_2d_min(points_norm, ref=HV_REF):
    if len(points_norm) == 0:
        return 0.0

    ref = np.array(ref, dtype=float)
    pts = points_norm[
        (points_norm[:, 0] <= ref[0]) &
        (points_norm[:, 1] <= ref[1])
    ]

    if len(pts) == 0:
        return 0.0

    pts = pts[np.argsort(pts[:, 0])]

    hv = 0.0
    prev_f2 = ref[1]

    for f1, f2 in pts:
        if f2 < prev_f2:
            hv += (ref[0] - f1) * (prev_f2 - f2)
            prev_f2 = f2

    return float(hv)

# ---------------- Common normalization bounds ----------------
# Use the same automatically calculated bounds as the convergence analysis
mins = objective_min
maxs = objective_max

print("HV normalization mins (f1 = 1-R², f2 = complexity):", mins)
print("HV normalization maxs (f1 = 1-R², f2 = complexity):", maxs)


# #>>> SENSITIVITY TEST <<<

# refs = [(1.0, 1.0), (1.05, 1.05), (1.10, 1.10)]

# for ref in refs:
#     hv_values = []

#     for pf in all_run_pareto_fronts:
#         pts = front_to_points_min(pf)
#         pts = unique_points(pts, decimals=10)
#         pts_norm = normalize_points(pts, mins, maxs)

#         hv = hypervolume_2d_min(
#             pts_norm,
#             ref=ref
#         )
#         hv_values.append(hv)

#     hv_values = np.array(hv_values)

#     print("\nReference point:", ref)
#     print("Mean HV:", hv_values.mean())
#     print("Std HV:", hv_values.std(ddof=1))
#     print("Min HV:", hv_values.min())
#     print("Max HV:", hv_values.max())

# ---------------- HV per run ----------------
hv_per_run = []

for run_idx, pf in enumerate(all_run_pareto_fronts, start=1):
    pts = front_to_points_min(pf)
    pts = unique_points(pts, decimals=10)
    pts_norm = normalize_points(pts, mins, maxs)
    hv = hypervolume_2d_min(pts_norm, ref=HV_REF)
    hv_per_run.append(hv)
    print(f"Run {run_idx:02d} HV: {hv:.6f}")

hv_per_run = np.array(hv_per_run)

print("\nHV summary (per run):")
print("Mean HV:", float(hv_per_run.mean()))
print("Std HV :", float(hv_per_run.std(ddof=1)) if len(hv_per_run) > 1 else 0.0)

# ---------------- HV for Final Pareto front ----------------
global_pts = front_to_points_min(global_pareto_front)
global_pts = unique_points(global_pts, decimals=10)
global_pts_norm = normalize_points(global_pts, mins, maxs)
hv_global = hypervolume_2d_min(global_pts_norm, ref=HV_REF)

print("\nFinal Pareto HV:", hv_global)

# ---------------- Plot HV analysis ----------------


# 1) HV Box plot


import os

fig, ax = plt.subplots(figsize=(5.6, 4.0), facecolor="white")

ax.boxplot(
    hv_per_run,
    widths=0.35,
    medianprops={"linewidth": 1.5},
    boxprops={"linewidth": 1.0},
    whiskerprops={"linewidth": 1.0},
    capprops={"linewidth": 1.0},
    flierprops={"markersize": 4}
)

ax.set_xticks([1])
ax.set_xticklabels(["BioSym"])
ax.set_ylabel("Hypervolume (HV)", labelpad=6)

ax.grid(axis="y", linestyle="--", linewidth=0.5, alpha=0.3)
ax.set_axisbelow(True)

fig.subplots_adjust(left=0.16,right=0.97,bottom=0.17,top=0.96)

boxplot_path = (
    "/content/drive/My Drive/GWL/r2_complexity/"
    "train219_val99_2obj of GP/Figure_pdf/hv_boxplot.pdf"
)

os.makedirs(os.path.dirname(boxplot_path), exist_ok=True)

fig.savefig(
    boxplot_path,
    format="pdf",
    #bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close(fig)


# 2) Histogram
plt.figure(figsize=(4.8, 3.6))

plt.hist(hv_per_run, bins=8, edgecolor="black", alpha=0.7)

plt.axvline(hv_per_run.mean(), color="red", linestyle="--",
            label=f"Mean = {hv_per_run.mean():.4f}")
plt.axvline(np.median(hv_per_run), color="green", linestyle=":",
            label=f"Median = {np.median(hv_per_run):.4f}")

plt.xlabel("Hypervolume")
plt.ylabel("Frequency")
plt.legend()

plt.grid(True, linestyle="--", linewidth=0.5, alpha=0.7)
plt.tight_layout()



plt.show()

# 3) Per-run performance
# ---------------- 3) HV per run ----------------

run_numbers = np.arange(1, len(hv_per_run) + 1)

fig, ax = plt.subplots(figsize=(5.6, 4.0), facecolor="white")

# HV of each independent run
ax.plot(
    run_numbers,
    hv_per_run,
    marker="o",
    markersize=4.5,
    linewidth=1.6,
    label="HV per run"
)

# Mean HV across the runs
ax.axhline(
    hv_per_run.mean(),
    linestyle="--",
    linewidth=1.2,
    color="black",
    label=f"Mean HV = {hv_per_run.mean():.4f}"
)

# HV of the combined final Pareto front
ax.axhline(
    hv_global,
    linestyle=":",
    linewidth=1.4,
    color="0.4",
    label=f"Final Pareto-front HV = {hv_global:.4f}"
)

ax.set_xlabel("Run number", labelpad=6)
ax.set_ylabel("Hypervolume (HV)", labelpad=6)

ax.set_xlim(1, len(hv_per_run))
ax.set_xticks([1, 5, 10, 15, 20])

# Preserve the range needed to show all values and reference lines
ymin = min(hv_per_run.min(), hv_global) - 0.01
ymax = max(hv_per_run.max(), hv_global) + 0.01
ax.set_ylim(ymin, ymax)

ax.grid(
    axis="y",
    linestyle="--",
    linewidth=0.5,
    alpha=0.3
)
ax.set_axisbelow(True)

ax.legend(
    loc="lower left",
    frameon=True,
    facecolor="white",
    edgecolor="0.8",
    fontsize=9
)

fig.subplots_adjust(left=0.16,right=0.97,bottom=0.17,top=0.96)

per_run_path = (
    "/content/drive/My Drive/GWL/r2_complexity/"
    "train219_val99_2obj of GP/Figure_pdf/hv_per_run.pdf"
)

fig.savefig(
    per_run_path,
    format="pdf",
    #bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close(fig)

# ---------------- Statistics ----------------
print("=" * 50)
print("HYPERVOLUME ANALYSIS FOR SYMBOLIC REGRESSION")
print("=" * 50)
print(f"Number of runs: {len(hv_per_run)}")
print(f"Mean HV: {hv_per_run.mean():.6f}")
print(f"Std Dev: {hv_per_run.std(ddof=1):.6f}" if len(hv_per_run) > 1 else "Std Dev: 0.000000")
print(f"Median: {np.median(hv_per_run):.6f}")
print(f"Min HV: {hv_per_run.min():.6f}")
print(f"Max HV: {hv_per_run.max():.6f}")
print(f"Range: {hv_per_run.max() - hv_per_run.min():.6f}")
print(f"Coefficient of Variation: {hv_per_run.std(ddof=1)/hv_per_run.mean():.4f}" if len(hv_per_run) > 1 else "Coefficient of Variation: 0.0000")
print(f"Final Pareto HV: {hv_global:.6f}")
print(f"Difference (Final vs Mean): {(hv_global / hv_per_run.mean() - 1) * 100:.2f}%")
print(f"Runs above mean HV: {sum(hv_per_run > hv_per_run.mean())}/{len(hv_per_run)}")

In [ ]:
## Multiple Linear Regression (MLR)


from sklearn.linear_model import LinearRegression

def build_linear_model_tree(intercept: float, coefs: np.ndarray) -> 'Node':
    """
    Convert: y = intercept + sum_i coefs[i] * x_i
    into a Node tree.
    """
    # start with intercept as constant node
    # We use 'Node' here - make sure your Node class exists!
    expr = Node(str(float(intercept)))

    for i, c in enumerate(coefs):
        # Creates (Weight * x0), (Weight * x1), etc.
        term = Node('*', [Node(str(float(c))), Node(f'x{i}')])
        expr = Node('+', [expr, term])  # Adds it to the total

    return expr


def linear_regression_to_txt(
    X: np.ndarray,
    y: np.ndarray,
    feature_names: list,
    out_txt_path: str,
    var_map: dict,
    precision: int = 6
):
    # 1. Create and train the model
    model = LinearRegression(fit_intercept=True)
    model.fit(X, y)

    # 2. See how accurate it is
    y_pred = model.predict(X)
    r2 = r2_score(y, y_pred)

    # 3. Build the math tree to calculate "complexity"
    tree = build_linear_model_tree(model.intercept_, model.coef_)
    complexity = tree.weighted_complexity()

    # 4. Turn the tree into a human-readable sentence
    eq_str = tree.to_readable_string(var_map)

    infix_eq_str = prefix_to_infix(eq_str) #prefix to infix

    # 5. Create the text report
    report = []
    report.append("Linear Regression")
    report.append(f"Features ({len(feature_names)}): {feature_names}")
    report.append("")
    # report.append("Expression:")
    # report.append(eq_str)
    # report.append("")

    report.append("Prefix Expression:")
    report.append(eq_str)
    report.append("")
    report.append("Infix Expression:")
    report.append(infix_eq_str)
    report.append("")
    report.append(f"R2: {r2:.{precision}f}")
    report.append(f"Weighted complexity: {complexity:.{precision}f}")

    report_text = "\n".join(report)

    # 6. Print to screen and save to file
    print(report_text)
    with open(out_txt_path, "w", encoding="utf-8") as f:
        f.write(report_text + "\n")

    return {
        "model": model,
        "r2": float(r2),
        "weighted_complexity": float(complexity),
        "tree": tree,
        "intercept": float(model.intercept_),
        "coef": model.coef_.astype(float),
        "txt_path": out_txt_path,
    }

In [ ]:
## linear regression expression write in the txt file

linear_txt_path = "/content/drive/My Drive/GWL/r2_complexity/train219_val99_2obj of GP/linear_regression.txt"

lin_result = linear_regression_to_txt(
    X=X,
    y=y,
    feature_names=FEATURE_COLUMNS,
    out_txt_path=linear_txt_path,
    var_map=symbolic_variable_map,
    precision=6
)

validation_path = "/content/drive/My Drive/GWL/r2_complexity/train219_val99_2obj of GP/validation_test99_dataset.csv"

validation_df = pd.read_csv(validation_path)

print("\nValidation dataset shape:", validation_df.shape)
print("Columns:", validation_df.columns.tolist())


# Select validation features in exactly the same order as training
X_val = validation_df[FEATURE_COLUMNS].to_numpy()
y_val = validation_df["Net Groundwater Recharge"].to_numpy()

# Predict using the existing training-fitted MLR model
y_val_pred = lin_result["model"].predict(X_val)

# Calculate validation R²
mlr_validation_r2 = r2_score(y_val, y_val_pred)

print(f"\nMLR training R²: {lin_result['r2']:.6f}")
print(f"MLR validation R²: {mlr_validation_r2:.6f}")
print(f"MLR weighted complexity: {lin_result['weighted_complexity']:.6f}")

In [ ]:
# MLR validation result add in .txt file and excel file

from pathlib import Path
import pandas as pd

# Save both files in the same folder as your MLR training report
output_dir = Path(linear_txt_path).parent

validation_txt_path = output_dir / "mlr_validation_r2.txt"
validation_excel_path = output_dir / "mlr_validation_results.xlsx"

# 1. Save validation R² to a text file
with open(validation_txt_path, "w", encoding="utf-8") as f:
    f.write("Multiple Linear Regression (MLR)\n")
    f.write(f"Training R²: {lin_result['r2']:.6f}\n")
    f.write(f"Validation R²: {mlr_validation_r2:.6f}\n")
    f.write(
        f"Weighted complexity: "
        f"{lin_result['weighted_complexity']:.6f}\n"
    )

# 2. Create an Excel file similar to your attached example
results_df = validation_df[FEATURE_COLUMNS].copy()
results_df["Actual"] = y_val
results_df["Predicted"] = y_val_pred

summary = (
    f"Training R²: {lin_result['r2']:.6f}, "
    f"Validation R²: {mlr_validation_r2:.6f}, "
    f"Complexity: {lin_result['weighted_complexity']:.6f}"
)

with pd.ExcelWriter(validation_excel_path, engine="openpyxl") as writer:
    results_df.to_excel(
        writer,
        sheet_name="MLR_Validation",
        index=False,
        startrow=2
    )

    worksheet = writer.sheets["MLR_Validation"]
    worksheet["A1"] = summary

print("Text file saved to:", validation_txt_path)
print("Excel file saved to:", validation_excel_path)

In [ ]:

# ==========================================
# BioSym Sensitivity Analysis: Setup
# ==========================================

import os
import pickle
import numpy as np
from datetime import datetime

# Original BioSym folder
BASE_DIR = (
    "/content/drive/My Drive/GWL/r2_complexity/"
    "train219_val99_2obj of GP"
)

# Check original convergence data
ORIGINAL_PKL = os.path.join(
    BASE_DIR,
    "BioSym_generation_pareto_fronts_20runs_for_convergence.pkl"
)

assert os.path.isfile(ORIGINAL_PKL), (
    "Original convergence file not found. Check the file path."
)

with open(ORIGINAL_PKL, "rb") as f:
    original_fronts = pickle.load(f)

print("Original independent runs:", len(original_fronts))
print("Generations in first run:", len(original_fronts[0]))

assert len(original_fronts) == 20
assert all(len(run) == 100 for run in original_fronts)

# One main folder for all sensitivity experiments
EXPERIMENT_DIR = os.path.join(
    BASE_DIR,
    "BioSym_Sensitivity_Analysis"
)

os.makedirs(EXPERIMENT_DIR, exist_ok=True)

print("\nSensitivity analysis folder:")
print(EXPERIMENT_DIR)


print("\nSetup completed successfully!")

In [ ]:
# ==========================================
# BioSym Sensitivity Analysis: Inspect original BioSym objective values
# ==========================================

# Check the structure of the saved Pareto fronts
print("First generation sample:")
print(original_fronts[0][0][:3])

print("\nFinal generation sample:")
print(original_fronts[0][-1][:3])

# Collect objective values from all runs and generations
original_points = np.array([
    point
    for run in original_fronts
    for front in run
    for point in front
], dtype=float)

# Verify the objective data
assert original_points.ndim == 2
assert original_points.shape[1] == 2
assert np.all(np.isfinite(original_points))

# BioSym objectives: minimize (1-R²) and complexity
original_f1 = 1.0 - original_points[:, 0]
original_f2 = original_points[:, 1]

print("\n--- Original Objective Ranges ---")
print("Total Pareto points:", len(original_points))

print("Minimum 1-R²:", original_f1.min())
print("Maximum 1-R²:", original_f1.max())

print("Minimum complexity:", original_f2.min())
print("Maximum complexity:", original_f2.max())

print("\nOriginal objective data verified successfully!")

In [ ]:
# ==========================================
# BioSym Sensitivity Analysis: Verify original BioSym HV
# ==========================================

import numpy as np

# Original objective normalization bounds
SENS_BASE_MIN = np.array([
    original_f1.min(),
    original_f2.min()
])

SENS_BASE_MAX = np.array([
    original_f1.max(),
    original_f2.max()
])

HV_REFERENCE = np.array([1.1, 1.1])

def sensitivity_hv(front, obj_min, obj_max):
    if len(front) == 0:
        return 0.0

    points = np.array([
        [1.0 - r2, complexity]
        for r2, complexity in front
    ], dtype=float)

    normalized = (points - obj_min) / (obj_max - obj_min)

    normalized = np.unique(normalized, axis=0)
    normalized = normalized[np.argsort(normalized[:, 0])]

    hv = 0.0
    current_y = HV_REFERENCE[1]

    for x, y in normalized:
        if x >= HV_REFERENCE[0] or y >= HV_REFERENCE[1]:
            continue

        if y < current_y:
            hv += (HV_REFERENCE[0] - x) * (current_y - y)
            current_y = y

    return float(hv)


# Calculate final-generation HV for all original runs
original_hv_values = np.array([
    sensitivity_hv(
        run[-1],
        SENS_BASE_MIN,
        SENS_BASE_MAX
    )
    for run in original_fronts
])

print("Original independent runs:", len(original_hv_values))
print("Original mean HV:", original_hv_values.mean())
print("Original HV standard deviation:",
      original_hv_values.std(ddof=1))

print("\nOriginal HV verification completed!")

In [ ]:
# =====================================================
# BioSym Sensitivity Analysis: FULL EXPERIMENT
# 4 crossover rates x 5 mutation rates x 10 runs
# =====================================================

import os
import pickle
import random
import numpy as np
from contextlib import redirect_stdout

# -----------------------------------------------------
# 1. Experiment settings
# -----------------------------------------------------

CROSSOVER_RATES = [0.5, 0.7, 0.9, 1.0]
MUTATION_RATES = [0.1, 0.3, 0.5, 0.7, 0.9]

NUM_RUNS = 10
POPULATION_SIZE = 100
GENERATIONS = 100
MAX_DEPTH = 6

# One fixed folder for all sensitivity-analysis results
EXPERIMENT_DIR = (
    "/content/drive/My Drive/GWL/r2_complexity/"
    "train219_val99_2obj of GP/"
    "BioSym_Sensitivity_Analysis"
)

os.makedirs(EXPERIMENT_DIR, exist_ok=True)

# -----------------------------------------------------
# 2. Verify an existing result file
# -----------------------------------------------------

def load_existing_result(file_path, pc, pm, seed):
    with open(file_path, "rb") as f:
        data = pickle.load(f)

    assert np.isclose(data["crossover_rate"], pc)
    assert np.isclose(data["mutation_rate"], pm)
    assert data["seed"] == seed
    assert data["population_size"] == POPULATION_SIZE
    assert data["generations"] == GENERATIONS
    assert len(data["generation_pareto_fronts"]) == GENERATIONS
    assert len(data["final_pareto_front"]) > 0

    return data

# -----------------------------------------------------
# 3. Save TXT version
# -----------------------------------------------------

def save_txt(data, txt_file):
    if os.path.exists(txt_file):
        return

    with open(txt_file, "x", encoding="utf-8") as f:
        f.write("BioSym Sensitivity Analysis\n")
        f.write("=" * 50 + "\n\n")

        f.write(f"Crossover probability: {data['crossover_rate']}\n")
        f.write(f"Mutation probability: {data['mutation_rate']}\n")
        f.write(f"Random seed: {data['seed']}\n")
        f.write(f"Population size: {data['population_size']}\n")
        f.write(f"Generations: {data['generations']}\n")
        f.write(f"Max depth: {data['max_depth']}\n\n")

        f.write("FINAL PARETO FRONT\n")
        f.write("-" * 50 + "\n")
        for i, (r2, complexity) in enumerate(data["final_pareto_front"], start=1):
            f.write(
                f"Solution {i}: R² = {r2:.6f}, Complexity = {complexity:.4f}\n"
            )

        f.write("\nGENERATION-WISE PARETO FRONTS\n")
        f.write("=" * 50 + "\n")
        for gen, front in enumerate(data["generation_pareto_fronts"], start=1):
            f.write(f"\nGeneration {gen}\n")
            for r2, complexity in front:
                f.write(f"R² = {r2:.6f}, Complexity = {complexity:.4f}\n")

# -----------------------------------------------------
# 4. Run all experiments
# -----------------------------------------------------

new_runs = 0
skipped_runs = 0
failed_runs = []

for pc in CROSSOVER_RATES:
    for pm in MUTATION_RATES:

        config_dir = os.path.join(
            EXPERIMENT_DIR,
            f"pc_{pc:.1f}_pm_{pm:.1f}"
        )
        os.makedirs(config_dir, exist_ok=True)

        for run_num in range(1, NUM_RUNS + 1):
            seed = 41 + run_num

            pkl_file = os.path.join(
                config_dir,
                f"run_{run_num:02d}_seed_{seed}.pkl"
            )
            txt_file = os.path.splitext(pkl_file)[0] + ".txt"

            # Skip already completed runs
            if os.path.isfile(pkl_file):
                try:
                    existing_data = load_existing_result(
                        pkl_file, pc, pm, seed
                    )
                    save_txt(existing_data, txt_file)
                    skipped_runs += 1
                    print(
                        f"Skipped existing: pc={pc:.1f}, pm={pm:.1f}, "
                        f"run={run_num}, seed={seed}",
                        flush=True
                    )
                    continue
                except Exception as e:
                    failed_runs.append(
                        f"Existing file problem: pc={pc}, pm={pm}, run={run_num}, seed={seed}, error={e}"
                    )
                    continue

            print(
                f"Starting: pc={pc:.1f}, pm={pm:.1f}, "
                f"run={run_num}, seed={seed}",
                flush=True
            )

            try:
                random.seed(seed)
                np.random.seed(seed)

                with open(os.devnull, "w") as quiet_output:
                    with redirect_stdout(quiet_output):
                        final_population, generation_fronts = run_gp_evolution(
                            X=X,
                            y=y,
                            feature_names=symbolic_variable_map,
                            population_size=POPULATION_SIZE,
                            generations=GENERATIONS,
                            max_depth=MAX_DEPTH,
                            mutation_rate=pm,
                            crossover_rate=pc,
                            collect_detailed_logs=False
                        )

                assert len(final_population) == POPULATION_SIZE
                assert len(generation_fronts) == GENERATIONS

                final_front = [
                    (float(ind.r2), float(ind.complexity))
                    for ind in get_pareto_front(final_population)
                ]

                assert len(final_front) > 0

                result_data = {
                    "crossover_rate": pc,
                    "mutation_rate": pm,
                    "seed": seed,
                    "population_size": POPULATION_SIZE,
                    "generations": GENERATIONS,
                    "max_depth": MAX_DEPTH,
                    "final_pareto_front": final_front,
                    "generation_pareto_fronts": generation_fronts
                }

                with open(pkl_file, "xb") as f:
                    pickle.dump(result_data, f)

                save_txt(result_data, txt_file)

                new_runs += 1

                print(
                    f"Completed: pc={pc:.1f}, pm={pm:.1f}, "
                    f"run={run_num}, seed={seed}",
                    flush=True
                )

            except Exception as e:
                failed_runs.append(
                    f"Failed: pc={pc}, pm={pm}, run={run_num}, seed={seed}, error={e}"
                )
                print(
                    f"Failed: pc={pc:.1f}, pm={pm:.1f}, "
                    f"run={run_num}, seed={seed}",
                    flush=True
                )

# -----------------------------------------------------
# 5. Final progress summary
# -----------------------------------------------------

completed = 0

for pc in CROSSOVER_RATES:
    for pm in MUTATION_RATES:
        config_dir = os.path.join(
            EXPERIMENT_DIR,
            f"pc_{pc:.1f}_pm_{pm:.1f}"
        )

        for run_num in range(1, NUM_RUNS + 1):
            seed = 41 + run_num
            pkl_file = os.path.join(
                config_dir,
                f"run_{run_num:02d}_seed_{seed}.pkl"
            )
            if os.path.isfile(pkl_file):
                completed += 1

print("\n===================================")
print("SENSITIVITY ANALYSIS PROGRESS")
print("===================================")
print("New runs completed:", new_runs)
print("Previously completed runs:", skipped_runs)
print("Total saved runs:", completed, "/ 200")
print("Remaining runs:", 200 - completed)
print("Failed runs:", len(failed_runs))

if failed_runs:
    print("\nFAILED RUN DETAILS:")
    for item in failed_runs:
        print(item)

print("\nResults folder:")
print(EXPERIMENT_DIR)

In [ ]:

# =====================================================
# BioSym Sensitivity Analysis:
# Step 2: Verify results and calculate objective ranges
# =====================================================

import os
import pickle
import numpy as np

EXPERIMENT_DIR = (
    "/content/drive/My Drive/GWL/r2_complexity/"
    "train219_val99_2obj of GP/"
    "BioSym_Sensitivity_Analysis"
)

CROSSOVER_RATES = [0.5, 0.7, 0.9, 1.0]
MUTATION_RATES = [0.1, 0.3, 0.5, 0.7, 0.9]

# Initialize objective bounds
SENS_MIN = np.array([np.inf, np.inf])
SENS_MAX = np.array([-np.inf, -np.inf])

verified_runs = 0
errors = []

# Check each saved sensitivity experiment
for pc in CROSSOVER_RATES:
    for pm in MUTATION_RATES:

        config_dir = os.path.join(
            EXPERIMENT_DIR,
            f"pc_{pc:.1f}_pm_{pm:.1f}"
        )

        for run_num in range(1, 11):

            seed = 41 + run_num

            pkl_file = os.path.join(
                config_dir,
                f"run_{run_num:02d}_seed_{seed}.pkl"
            )

            try:
                with open(pkl_file, "rb") as f:
                    data = pickle.load(f)

                # Verify experiment settings
                assert np.isclose(data["crossover_rate"], pc)
                assert np.isclose(data["mutation_rate"], pm)
                assert data["seed"] == seed
                assert data["population_size"] == 100
                assert data["generations"] == 100

                # Verify saved Pareto fronts
                assert len(data["generation_pareto_fronts"]) == 100
                assert len(data["final_pareto_front"]) > 0

                # Inspect all generations
                for front in data["generation_pareto_fronts"]:

                    points = np.asarray(front, dtype=float)

                    assert points.ndim == 2
                    assert points.shape[1] == 2
                    assert len(points) > 0
                    assert np.all(np.isfinite(points))

                    # Convert to minimization objectives
                    objectives = np.column_stack((
                        1.0 - points[:, 0],
                        points[:, 1]
                    ))

                    # Update common normalization bounds
                    SENS_MIN = np.minimum(
                        SENS_MIN,
                        objectives.min(axis=0)
                    )

                    SENS_MAX = np.maximum(
                        SENS_MAX,
                        objectives.max(axis=0)
                    )

                # Verify final Pareto front
                final_points = np.asarray(
                    data["final_pareto_front"], dtype=float
                )

                assert final_points.ndim == 2
                assert final_points.shape[1] == 2
                assert np.all(np.isfinite(final_points))

                verified_runs += 1

            except Exception as e:
                errors.append(f"{pkl_file}: {e}")


# =====================================================
# Verification summary
# =====================================================

print("===================================")
print("SENSITIVITY DATA VERIFICATION")
print("===================================")

print("Verified runs:", verified_runs, "/ 200")
print("Errors:", len(errors))

if errors:
    for error in errors[:10]:
        print(error)
    raise RuntimeError(
        "Data verification failed. Resolve the errors before calculating HV."
    )

assert verified_runs == 200
assert np.all(SENS_MAX > SENS_MIN)

print("\n--- Objective ranges ---")

print("Minimum 1-R²:", SENS_MIN[0])
print("Maximum 1-R²:", SENS_MAX[0])

print("Minimum complexity:", SENS_MIN[1])
print("Maximum complexity:", SENS_MAX[1])

print("\nAll sensitivity results verified successfully!")

In [ ]:

# =====================================================
# BioSym Sensitivity Analysis
# Step 3: Calculate HV for all 200 runs
# =====================================================

import os
import pickle
import numpy as np
import pandas as pd

HV_REF = np.array([1.1, 1.1], dtype=float)

# Use the common normalization bounds
# calculated in the previous verification cell.
OBJ_MIN = SENS_MIN.copy()
OBJ_MAX = SENS_MAX.copy()

assert np.all(OBJ_MAX > OBJ_MIN)


# -----------------------------------------------------
# 1. Hypervolume calculation
# -----------------------------------------------------

def calculate_sensitivity_hv(front):

    if len(front) == 0:
        return 0.0

    # Convert objectives to minimization
    points = np.array([
        [1.0 - r2, complexity]
        for r2, complexity in front
    ], dtype=float)

    # Normalize using common bounds
    normalized = (points - OBJ_MIN) / (OBJ_MAX - OBJ_MIN)

    # Remove duplicate points
    normalized = np.unique(normalized, axis=0)

    # Sort by the first objective
    normalized = normalized[
        np.argsort(normalized[:, 0])
    ]

    hv = 0.0
    current_y = HV_REF[1]

    for x, y in normalized:

        if x >= HV_REF[0] or y >= HV_REF[1]:
            continue

        if y < current_y:
            hv += (
                (HV_REF[0] - x)
                * (current_y - y)
            )
            current_y = y

    return float(hv)


# -----------------------------------------------------
# 2. Calculate HV for all sensitivity runs
# -----------------------------------------------------

results = []

for pc in CROSSOVER_RATES:

    for pm in MUTATION_RATES:

        hv_values = []

        config_dir = os.path.join(
            EXPERIMENT_DIR,
            f"pc_{pc:.1f}_pm_{pm:.1f}"
        )

        for run_num in range(1, 11):

            seed = 41 + run_num

            pkl_file = os.path.join(
                config_dir,
                f"run_{run_num:02d}_seed_{seed}.pkl"
            )

            with open(pkl_file, "rb") as f:
                data = pickle.load(f)

            # Calculate HV using final Pareto front
            hv = calculate_sensitivity_hv(
                data["final_pareto_front"]
            )

            hv_values.append(hv)

        # Verify 10 independent runs
        assert len(hv_values) == 10

        results.append({
            "Crossover": pc,
            "Mutation": pm,
            "Mean_HV": np.mean(hv_values),
            "Std_HV": np.std(hv_values, ddof=1),
            "Min_HV": np.min(hv_values),
            "Max_HV": np.max(hv_values),
            "Runs": len(hv_values)
        })


# -----------------------------------------------------
# 3. Create sensitivity summary table
# -----------------------------------------------------

sensitivity_summary = pd.DataFrame(results)

assert len(sensitivity_summary) == 20

print("\nBioSym Sensitivity Analysis Results")
print("=" * 65)

print(
    sensitivity_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# -----------------------------------------------------
# 4. Save results as CSV
# -----------------------------------------------------

summary_file = os.path.join(
    EXPERIMENT_DIR,
    "BioSym_Sensitivity_HV_Summary.csv"
)

# Protect previously saved analysis files
if os.path.exists(summary_file):
    raise FileExistsError(
        f"Summary file already exists: {summary_file}"
    )

sensitivity_summary.to_csv(
    summary_file,
    index=False
)

print("\nHV calculation completed!")
print("Parameter combinations:", len(sensitivity_summary))
print("Total independent runs:", sensitivity_summary["Runs"].sum())
print("Summary saved:", summary_file)

In [ ]:

# =====================================================
# BioSym Sensitivity Analysis: Journal-Style HV Heatmap
# Inspired by the presentation style of Nayeem et al.
# =====================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------------------------------
# 1. Load the existing sensitivity-analysis results
# -----------------------------------------------------

EXPERIMENT_DIR = (
    "/content/drive/My Drive/GWL/r2_complexity/"
    "train219_val99_2obj of GP/"
    "BioSym_Sensitivity_Analysis"
)

summary_file = os.path.join(
    EXPERIMENT_DIR,
    "BioSym_Sensitivity_HV_Summary.csv"
)

assert os.path.isfile(summary_file), (
    "Sensitivity HV summary CSV file not found."
)

df = pd.read_csv(summary_file)

# -----------------------------------------------------
# 2. Prepare the 5 x 4 heatmap
# -----------------------------------------------------

crossover_rates = [0.5, 0.7, 0.9, 1.0]
mutation_rates = [0.1, 0.3, 0.5, 0.7, 0.9]

# Verify the experimental data
assert len(df) == 20
assert df["Runs"].eq(10).all()

assert not df.duplicated(
    subset=["Crossover", "Mutation"]
).any()

heatmap_data = df.pivot(
    index="Mutation",
    columns="Crossover",
    values="Mean_HV"
)

heatmap_data = heatmap_data.reindex(
    index=mutation_rates,
    columns=crossover_rates
)

assert heatmap_data.shape == (5, 4)
assert np.isfinite(heatmap_data.values).all()

values = heatmap_data.values

# -----------------------------------------------------
# 3. Set journal-style figure properties
# -----------------------------------------------------

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 10,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})

fig, ax = plt.subplots(
    figsize=(6.2, 4.8),
    facecolor="white"
)

# -----------------------------------------------------
# 4. Create the heatmap
# -----------------------------------------------------

im = ax.imshow(
    values,
    cmap="Reds",
    interpolation="nearest",
    aspect="equal",
    origin="lower",
    vmin=values.min(),
    vmax=values.max()
)

# -----------------------------------------------------
# 5. Set axis labels and ticks
# -----------------------------------------------------

ax.set_xticks(np.arange(len(crossover_rates)))

ax.set_xticklabels([
    f"{p:.1f}" for p in crossover_rates
])

ax.set_yticks(np.arange(len(mutation_rates)))

ax.set_yticklabels([
    f"{p:.1f}" for p in mutation_rates
])

ax.set_xlabel(
    r"Crossover probability ($p_c$)",
    labelpad=8
)

ax.set_ylabel(
    r"Mutation probability ($p_m$)",
    labelpad=8
)

# -----------------------------------------------------
# 6. Add thin white gridlines
# -----------------------------------------------------

ax.set_xticks(
    np.arange(-0.5, len(crossover_rates), 1),
    minor=True
)

ax.set_yticks(
    np.arange(-0.5, len(mutation_rates), 1),
    minor=True
)

ax.grid(
    which="minor",
    color="white",
    linewidth=1.2
)

ax.tick_params(
    which="minor",
    bottom=False,
    left=False
)

ax.tick_params(
    axis="both",
    which="major",
    length=0
)

# Remove unnecessary outer borders
for spine in ax.spines.values():
    spine.set_visible(False)

# -----------------------------------------------------
# 7. Display mean HV inside every cell
# -----------------------------------------------------

for i in range(len(mutation_rates)):
    for j in range(len(crossover_rates)):

        value = values[i, j]

        # Determine text color from background brightness
        rgba = im.cmap(im.norm(value))

        brightness = (
            0.299 * rgba[0]
            + 0.587 * rgba[1]
            + 0.114 * rgba[2]
        )

        text_color = (
            "white" if brightness < 0.55
            else "black"
        )

        ax.text(
            j,
            i,
            f"{value:.4f}",
            ha="center",
            va="center",
            color=text_color,
            fontsize=10
        )

# -----------------------------------------------------
# 8. Add colorbar
# -----------------------------------------------------

cbar = fig.colorbar(
    im,
    ax=ax,
    fraction=0.046,
    pad=0.04
)

cbar.set_label(
    "Mean hypervolume (HV)",
    rotation=90,
    labelpad=10,
    fontsize=10
)

cbar.ax.tick_params(
    labelsize=9,
    length=3
)

cbar.outline.set_linewidth(0.5)

# -----------------------------------------------------
# 9. Save as a vector PDF
# -----------------------------------------------------

pdf_dir = (
    "/content/drive/My Drive/GWL/r2_complexity/"
    "train219_val99_2obj of GP/"
    "Figure_pdf"
)

os.makedirs(pdf_dir, exist_ok=True)

pdf_file = os.path.join(
    pdf_dir,
    "BioSym_Sensitivity_HV_Heatmap.pdf"
)

fig.savefig(
    pdf_file,
    format="pdf",
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close(fig)

print("Journal-style heatmap saved successfully!")
print("File:", pdf_file)